# LLM Fiscal Policy Classification Demo

Demonstrates fiscal and monetary policy classification prompts with 4 variants each: simple, with_definitions, chain_of_thought, and few_shot.

In [1]:
# Setup
from dotenv import load_dotenv
load_dotenv('../../.env')
import os, sys
sys.path.insert(0, '../../libs')
sys.path.insert(0, '../../src/Traction/prompts')
sys.path.insert(0, '../../src/Traction')
import config

from llm_factory_openai import LLMAgent
# Use _process_batch_async to process batch_messages
import asyncio
from llm_batch_process_utils import _process_batch_async
from llm_factory_openai import BatchAsyncLLMAgent
from llm_batch_process_utils import _merge_ids_with_responses

from prompt_utils import load_prompt, format_messages
from schema import PROMPT_REGISTRY
from pathlib import Path
import pandas as pd
from llm_batch_process_utils import _build_batch_messages_from_df_flexible
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, precision_score, recall_score

#### Set up llm

In [2]:
# Initialize LLM Agent
llm_agent = LLMAgent(
    api_key=os.getenv('OPENAI_API_KEY'),
    model='gpt-5-nano',
    temperature=1
)

#### Load prompts templates

In [3]:
# Group prompts by category
prompts = {'monetary_stance': [], 'monetary_agreement': [], 'fiscal_stance': [], 'fiscal_agreement': []}
for key, entry in PROMPT_REGISTRY.items():
    fname = entry["prompt_file"]
    for category in prompts.keys():
        if fname.startswith(category):
            prompts[category].append(key)
            break

print(f"Loaded {sum(len(v) for v in prompts.values())} prompts across {len(prompts)} categories")

Loaded 16 prompts across 4 categories


In [4]:
prompts

{'monetary_stance': ['monetary_stance_simple',
  'monetary_stance_with_definitions',
  'monetary_stance_few_shot',
  'monetary_stance_chain_of_thought'],
 'monetary_agreement': ['monetary_agreement_simple',
  'monetary_agreement_with_definitions',
  'monetary_agreement_few_shot',
  'monetary_agreement_chain_of_thought'],
 'fiscal_stance': ['fiscal_stance_simple',
  'fiscal_stance_with_definitions',
  'fiscal_stance_few_shot',
  'fiscal_stance_chain_of_thought'],
 'fiscal_agreement': ['fiscal_agreement_simple',
  'fiscal_agreement_with_definitions',
  'fiscal_agreement_few_shot',
  'fiscal_agreement_chain_of_thought']}

## Load data

In [5]:
data_dir = Path(config.data_dir / 'Finetuning_data')
data_dir.exists()

True

In [6]:
# prepare files for different tasks
df_train = pd.read_excel( data_dir / 'Fiscal/cv/training_2.xlsx')
df_test = pd.read_excel(data_dir / 'Fiscal/cv/testing_2.xlsx')

df_train_agree = df_train[['index', 'Print ISBN', 'staff', 'buff', 'country', 'year', 'agreement_general', 'disagreement_areas']]
df_test_agree = df_test[['index', 'Print ISBN', 'staff', 'buff', 'country', 'year', 'agreement_general', 'disagreement_areas']]

In [7]:
for col in ['agreement_general', 'disagreement_areas']:
    print(df_train_agree[col].value_counts(normalize=True))
    print(df_test_agree[col].value_counts(normalize=True))
    print('\n')

agreement_general
mostly agree           0.554167
disagreement exists    0.300000
irrelevant             0.145833
Name: proportion, dtype: float64
agreement_general
mostly agree           0.616667
disagreement exists    0.216667
irrelevant             0.166667
Name: proportion, dtype: float64


disagreement_areas
government debt & financing                                0.223684
government revenue                                         0.197368
near-term policy direction                                 0.144737
government expenditure                                     0.131579
economic fundamentals                                      0.105263
fiscal framework                                           0.039474
government expenditure; near-term policy direction         0.026316
medium-term fiscal stance                                  0.026316
government revenue; near-term policy direction             0.026316
government debt & financing; near-term policy direction    0.026316
econo

In [8]:
# Demo: Build batch messages for monetary agreement using flexible function
prompt_key = 'fiscal_agreement_few_shot'
registry_entry = PROMPT_REGISTRY[prompt_key]
prompt_file = registry_entry["prompt_file"]
prompt_dir = Path('../../src/Traction/prompts')
# Load prompt template
prompt_template = load_prompt(prompt_dir / prompt_file).sections
response_model = registry_entry["response_model"]

In [9]:
prompt_template

{'system': 'You are an experienced macroeconomist from IMF. Given two pieces of texts expressing the views of IMF staff and a country\'s authorities, respectively, your task is to determine whether the authorities agree or disagree with IMF staff on issues related to the country\'s fiscal policy.\n\n**First**, assign a value to the "agreement" field: "mostly agree"/"disagreement exists"/"irrelevant". Note that the authorities\' agreement with IMF staff\'s views is different in concept from IMF staff\'s agreement with the authorities\' views. If the two pieces of texts discuss common aspect(s) of fiscal policy or if the authorities directly express agreement/disagreement to fiscal related issues in either text:\n(a) if the authorities disagree with IMF staff on any fiscal policy issues, assign "disagreement exists";\n(b) if there is no disagreement by the authorities, assign "mostly agree";\n(c) if the authorities do not directly express agreement/disagreement with IMF staff on fiscal r

In [10]:
# Define column mapping: template placeholder -> dataframe column
# Template uses: {COUNTRY}, {YEAR}, {STAFF_TEXT}, {AUTHORITY_TEXT}
# DataFrame has: 'country', 'year', 'staff', 'buff'
column_mapping = {
    'COUNTRY': 'country',
    'YEAR': 'year',
    'STAFF_TEXT': 'staff',
    'AUTHORITY_TEXT': 'buff'
}

In [11]:
# combine both traing and testing data
sample_df = pd.concat([df_train_agree, df_test_agree])
#sample_df = df_train_agree

In [12]:
# Build batch messages from first 5 rows
batch_messages, batch_ids = _build_batch_messages_from_df_flexible(
    sample_df,
    prompt_template,
    column_mapping=column_mapping,
    id_column='index',
    max_text_length=8000
)

In [13]:
print(f"Generated {len(batch_messages)} batch messages")
print(f"Sample IDs: {batch_ids}")
print(f"\nFirst message structure:")
print(f"Number of message parts: {len(batch_messages[0])}")
for i, msg in enumerate(batch_messages[30][:4]):  # Show first 2 parts
    print(f"\nPart {i+1} - Role: {msg.get('role', 'N/A')}")
    content = msg.get('content', '')
    print(f"{content[:2500]}...")


Generated 300 batch messages
Sample IDs: ['200', '43', '159', '59', '167', '56', '5', '50', '299', '247', '114', '3', '88', '4', '13', '131', '111', '44', '79', '271', '70', '280', '58', '289', '292', '154', '26', '197', '57', '235', '285', '138', '148', '108', '296', '118', '218', '134', '21', '205', '40', '150', '168', '105', '183', '253', '34', '282', '287', '84', '286', '35', '295', '290', '87', '120', '275', '72', '178', '128', '266', '273', '121', '139', '39', '210', '133', '181', '222', '203', '177', '297', '207', '202', '1', '64', '277', '166', '142', '116', '98', '117', '173', '198', '28', '89', '115', '196', '7', '232', '129', '221', '162', '270', '169', '293', '201', '180', '123', '206', '257', '241', '184', '240', '190', '175', '32', '144', '96', '204', '69', '283', '272', '281', '17', '284', '93', '176', '239', '109', '160', '195', '186', '140', '127', '228', '164', '119', '14', '83', '20', '276', '11', '51', '248', '124', '80', '174', '233', '55', '255', '37', '172', '42'

In [14]:
try:
    response = llm_agent.get_response_content(batch_messages[15], reasoning_effort='low', response_format=response_model)
    print(response)
except Exception as e:
    print(f"Error: {e}")

agreement='mostly agree' disagreement_areas=''


In [15]:
batch_agent = BatchAsyncLLMAgent(
    api_key=os.getenv('OPENAI_API_KEY'),
    model='gpt-5',
    temperature=1
)


In [16]:
# Process a small subset to demo
subset_messages = batch_messages
subset_ids = batch_ids

async def run_batch():
    results = await _process_batch_async(
        batch_agent,
        subset_messages,
        response_model=registry_entry["response_model"],
        batch_size=8,
        max_completion_tokens=2000,
        reasoning_effort='low'
    )
    return results

results = asyncio.run(run_batch())

Processing batches: 100%|██████████| 300/300 [06:12<00:00,  1.24s/msg]


In [17]:
# Show merged outputs (id + parsed response)
merged = _merge_ids_with_responses(subset_ids, results)
print(f"Processed {len(merged)} responses")
print(merged[0])

Processed 300 responses
{'agreement': 'irrelevant', 'disagreement_areas': '', 'id': '200'}


In [18]:
# Convert merged results to DataFrame
df_merged = pd.DataFrame(merged)
# Add _llm suffix to column names from the LLM results
llm_columns = [col for col in df_merged.columns if col not in ['id']]
rename_dict = {col: f"{col}_llm" for col in llm_columns}
df_merged = df_merged.rename(columns=rename_dict)
df_merged

,agreement_llm,disagreement_areas_llm,id
0,irrelevant,,200
1,mostly agree,[],43
2,disagreement exists,"['near-term policy direction', 'government rev...",159
3,disagreement exists,"['near-term policy direction', 'government def...",59
4,mostly agree,,167
...,...,...,...
295,mostly agree,,9
296,mostly agree,,54
297,mostly agree,[],19
298,mostly agree,,165


In [19]:
# Fix the merge issue by converting id column types to match
df_merged['id'] = df_merged['id'].astype(int)
df_merged = df_merged.merge(sample_df, left_on='id', right_on='index', how='left')
# Keep only records with no NaN values in key columns
df_merged = df_merged.dropna(subset=['agreement_general', 'agreement_llm'])
print(len(df_merged))
df_merged.head()

300


,agreement_llm,disagreement_areas_llm,id,index,Print ISBN,staff,buff,country,year,agreement_general,disagreement_areas
0,irrelevant,,200,200,9798400231247,38. The 2023 Budget strikes the right balance ...,6. Morocco’s Minister of Finance and BAM signe...,Morocco,2022.0,irrelevant,NaN
1,mostly agree,[],43,43,9781484335918,"37. In 2015-16, the Papua New Guinea (PNG) eco...","In response to these challenges, the newly-ele...",Papua New Guinea,2017.0,mostly agree,NaN
2,disagreement exists,"['near-term policy direction', 'government rev...",159,159,9781498320993,56. The U.S. public debt is on an unsustainabl...,Staff’s analysis of the tax reform misses impo...,United States,2019.0,irrelevant,economic fundamentals
3,disagreement exists,"['near-term policy direction', 'government def...",59,59,9781475575620,33. The authorities have taken policy steps to...,The fiscal policy stance for 2016 remained res...,Angola,2016.0,mostly agree,NaN
4,mostly agree,,167,167,9798400256899,43. Fiscal policy has ample space available to...,Fiscal policy is disciplined and a follows a c...,Vietnam,2023.0,mostly agree,NaN


In [20]:
# query v6:
print("accuracy Score:")
print(accuracy_score(df_merged['agreement_general'], 
               df_merged['agreement_llm']))

print("f1 Score:")      
print(f1_score(df_merged['agreement_general'], 
         df_merged['agreement_llm'], 
         average='weighted'))

print("Confusion Matrix:")
confusion_matrix(df_merged['agreement_general'], 
                 df_merged['agreement_llm'], 
                 labels=['disagreement exists', 
                         'mostly agree', 
                         'irrelevant'])

accuracy Score:
0.7133333333333334
f1 Score:
0.6635031708506886
Confusion Matrix:


array([[ 63,  22,   0],
       [ 21, 149,   0],
       [  6,  37,   2]])

In [60]:
import os,sys
sys.path.append('..')
import config
from openai import OpenAI
import json

directory = config.data_dir

In [61]:
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

In [62]:
df = pd.read_excel(directory / 'Finetuning_data/labelled_fiscal_v2.xlsx')
df_agree = df[['index', 'Print ISBN', 'staff', 'buff', 'country', 'year', 'agreement_general', 'disagreement_areas']]

In [63]:
# 3. Few-shot
# version 4: adding a few examples
example_l = [
    '''Example 1:\nCountry: Philippines; Year: 2018\n\nPart1 - IMF staff:\n35. To balance growth and stability objectives, the authorities should adopt a neutral fiscal stance over 2018–2019. This implies an overall deficit of 2.4 percent in 2018 and 2.5 percent of GDP in 2019 (compared to staff’s current baseline of 2.8 and 3.2 percent), which would support pro-growth infrastructure investment without overburdening monetary policy, while containing inflationary pressures. Raising tax revenues and reallocating spending from nonpriority programs can support the continued expansion of public investment and social spending.\n\nPart2 - Authority:\nAuthorities are keeping their expansionary fiscal policy, as originally programmed. While they have carefully considered the staff policy recommendation to keep a neutral fiscal stance to balance growth and stability, Authorities continue to see the imperative for stronger investment in infrastructure. ... Part of staff argument for a neutral fiscal stance was to “limit overheating risks” and avoid “overburdening monetary policy” While Authorities are cognizant of elevated risks, they continue to see few signs of possible overheating in the economy.\n\nAnswer: {"agreement": "disagreement exists", "disagreement_areas": "['economic fundamentals', 'near-term policy direction']"}.'''.replace('\n\n','\n'),
    '''Example 2:\nCountry: India; Year: 2017\n\nPart1 - IMF staff:\n55. The authorities should press ahead with their longer-term objective of substantially reducing fiscal deficits and the public debt burden. With continued delays in reaching medium-term deficit targets, India’s public debt ratio is likely to remain high and fiscal policy space remains limited. ...\n\nPart2 - Authority:\n5. We want to emphasize the commitment of our authorities on medium-term fiscal consolidation. A conducive environment for improvement in state public finances has been created through suitable incentives/costs for states to renew fiscal efforts towards consolidation. ... As regards debt, India’s public debt is sustainable both because of the authorities’ commitments for fiscal consolidation and the projected interest versus growth trajectory going forward.\n\nAnswer: {"agreement": "disagreement exists", "disagreement_areas": "['government debt & financing']"}.'''.replace('\n\n','\n'),
    '''Example 3:\nCountry: Denmark; Year: 2019\n\nPart1 - IMF staff:\n47. Denmark’s public finances are sound with substantial fiscal space in the medium term. The fiscal stance should remain neutral, while letting automatic stabilizers operate fully in case of shocks to aggregate demand. In the event of a severe downturn, additional temporary loosening should be considered, while remaining anchored to the medium-term objective. Efficiency-improving reforms that cover both revenues and expenditures could be implemented in a fiscally-neutral way or designed to provide stimulus if loosening is warranted.\n\nPart2 - Authority:\nFor more than two decades, Danish fiscal policy has been conducted within a forward-looking medium-term fiscal framework. The associated plans contain operational targets for the medium-term structural fiscal balance and play an important role in ensuring long-term fiscal sustainability. The most recent update of the 2025-plan aims at structural fiscal balance in 2025.\n\nAnswer: {"agreement": "mostly agree", "disagreement_areas": "[]"}.'''.replace('\n\n','\n'),
]

for i, row in df_agree.iterrows():
    chat_completion = client.chat.completions.create(
            messages=[
                {   "role": "system",
                    "content": '''You are an experience macroeconomist from IMF. Given two pieces of texts expressing the views of IMF staff and a country's authorities, respectively, your task is to determine whether the authorities agree or disagree with IMF staff on issues related to the country's fiscal policy. First, assign a value to the "agreement" field": "mostly agree"/"disagreement exists"/"irrelevant". Note that the authorities' agreement with IMF staff's views is different in concept from IMF staff's agreement with the authorities' views. If the two pieces of texts discuss common aspect(s) of fiscal policy or if the authorities directly express agreement/disagreement to fiscal related issues in either text: (a) if the authorities disagree with IMF staff on any fiscal policy issues, assign "disagreement exists"; (b) if there is no disagreement by the authorities, assign "mostly agree"; (c) if the authorities do not directly express agreement/disagreement with IMF staff on fiscal related issues, and either of the texts does not discuss fiscal policy or they discuss entirely different aspects of fiscal policy, assign "irrelevant".  Second, if disagreement exists, summarize the area(s) of disagreement in short phrase(s) and list them in the "disagreement_areas" field; for example, "near-term policy direction", "fiscal revenue", "fiscal expenditure", "government debt & financing", "economic assessment", "fiscal framework", "medium-term fiscal stance", etc; if the authorities do not disagree with staff, leave the "disagreement_areas" field blank.\n\n%s\n\nReturn a JSON dict without additional texts: \"agreement\", \"disagreement_areas\".'''%('\n\n'.join(example_l)),},
                {   "role": "user",
                    "content":  '''Country: %s; Year: %s\n\nPart1 - IMF staff:\n%s\n\nPart2 - Authority:\n%s''' % (row['country'], row['year'], row['staff'], row['buff'])}
        ],
            model='gpt-5',
            temperature=1
        )
    try:
        result = json.loads(chat_completion.choices[0].message.content.replace('```json','').replace('```','').replace('\n',' '))
        df_agree.loc[i,'agreement_gpt4'] = result['agreement']
        df_agree.loc[i,'disagreement_areas_gpt4'] = result['disagreement_areas']
    except Exception:
        df_agree.loc[i, 'error_gpt4'] = chat_completion.choices[0].message.content
        

# query v4:
print(accuracy_score(df_agree['agreement_general'], df_agree['agreement_gpt4']), f1_score(df_agree['agreement_general'], df_agree['agreement_gpt4'], average='weighted'), confusion_matrix(df_agree['agreement_general'], df_agree['agreement_gpt4'], labels=['disagreement exists', 'mostly agree', 'irrelevant']))


{"agreement": "mostly agree", "disagreement_areas": []}
{"agreement": "disagreement exists", "disagreement_areas": ["near-term policy direction", "fiscal framework", "fiscal expenditure"]}
{"agreement": "disagreement exists", "disagreement_areas": ["government debt & financing", "fiscal framework"]}
{"agreement": "disagreement exists", "disagreement_areas": ["economic assessment", "fiscal expenditure"]}
{"agreement": "disagreement exists", "disagreement_areas": ["social safety net design", "fiscal expenditure"]}
{"agreement": "mostly agree", "disagreement_areas": []}
{"agreement": "mostly agree", "disagreement_areas": []}
{"agreement":"mostly agree","disagreement_areas":[]}
{"agreement": "mostly agree", "disagreement_areas": []}
{"agreement": "mostly agree", "disagreement_areas": []}
{"agreement": "mostly agree", "disagreement_areas": []}
{"agreement": "disagreement exists", "disagreement_areas": ["fiscal expenditure", "tax policy"]}
{"agreement": "mostly agree", "disagreement_areas": 

KeyboardInterrupt: 

#### Try Stance prediction

In [21]:
# Demo: Build batch messages for monetary agreement using flexible function
prompt_key = 'fiscal_stance_few_shot'
prompt_dir = Path('../../src/Traction/prompts')
registry_entry = PROMPT_REGISTRY[prompt_key]
prompt_file = registry_entry["prompt_file"]
prompt_template = load_prompt(prompt_dir / prompt_file).sections
response_model = registry_entry["response_model"]

In [22]:
prompt_template

{'system': 'You are an experience macroeconomist from IMF. Given a piece of text concerning a particular country in a given year expressing the views of {TYPE}, classify the {TYPE_POSSESSIVE} {VERB} near-term (next year) direction of change in fiscal policy stance as described in the text into "tightening"/"tightening bias"/"no change"/"loosening bias"/"loosening"/"unclear"/"irrelevant". {EXPLANATION}\n\nDefinitions:\nTightening: Suggests a plan to reduce fiscal deficits or move towards a surplus. This can be achieved through higher taxation or non-tax revenues, reduced government spending, or a combination of both.\nTightening Bias: Indicates a leaning towards a tightening fiscal policy but without a full commitment.\nNo Change: Indicates a plan to maintain the current fiscal policy stance without significant changes to overall fiscal balance.\nLoosening Bias: Suggests a leaning towards adopting a loosening fiscal policy, though not explicitly planning to do so.\nLoosening: Suggests a

In [23]:

# Rename columns from stance_near_term to stance_future
df_stance = pd.concat([df_train, df_test], ignore_index=True)
df_stance = df_stance.rename(columns={
    'staff_stance_near_term': 'staff_stance_future',
    'buff_stance_near_term': 'buff_stance_future',
    'agreement_stance_near_term_num': 'agreement_stance_future_num'
})

In [24]:
df_stance_temp = pd.DataFrame()
for tp in ['staff', 'buff']:
    df_temp = df_stance[['index', 'Print ISBN', 'country', 'year', tp, '%s_stance_future'%tp]]
    df_temp = df_temp.rename(columns={tp: 'text'}).rename(columns={c: c.replace(tp+'_', '') for c in df_temp.columns})
    df_temp['type'] = tp
    df_stance_temp = pd.concat([df_stance_temp, df_temp], ignore_index=True)

df_stance = df_stance_temp

In [25]:
for col in ['stance_future']:
    print(df_stance[col].value_counts(normalize=True))
    print('\n')

stance_future
tightening         0.546667
loosening          0.136667
unclear            0.111667
no change          0.088333
tightening bias    0.070000
loosening bias     0.046667
Name: proportion, dtype: float64




In [26]:
df_stance.head()

,index,Print ISBN,country,year,text,stance_future,type
0,200,9798400231247,Morocco,2022.0,38. The 2023 Budget strikes the right balance ...,tightening,staff
1,43,9781484335918,Papua New Guinea,2017.0,"37. In 2015-16, the Papua New Guinea (PNG) eco...",tightening,staff
2,159,9781498320993,United States,2019.0,56. The U.S. public debt is on an unsustainabl...,tightening,staff
3,59,9781475575620,Angola,2016.0,33. The authorities have taken policy steps to...,tightening,staff
4,167,9798400256899,Vietnam,2023.0,43. Fiscal policy has ample space available to...,loosening bias,staff


In [27]:
example_dict = {'staff': 'Example 1: Country: Tunisia; Year: 2015; Text: \"The modest fiscal loosening in 2015 appropriately responds to weaker economic activity. Going forward, fiscal consolidation is needed to reduce external imbalances, restore private sector confidence, and ensure debt sustainability.\" Answer: \"tightening\".\nExample 2: Country: Denmark; Year: 2019; Text: \"The fiscal stance should remain neutral, while letting automatic stabilizers operate fully in case of shocks to aggregate demand.\" Answer: \"no change\".\nExample 3: Country: Denmark; Year: 2017; Text: \"Thus, provided that strong new labor market reforms are agreed to raise labor supply, it would be appropriate to slow the pace of consolidation somewhat to facilitate cuts in the high tax burden.\" Answer: \"loosening bias\".',
               'buff': 'Example 1: Country: Tunisia; Year: 2015; Text: \"The authorities are firmly committed to take additional measures to attain their fiscal objectives in 2015, including through reduction of non-essential non-wage expenditure. They are committed to fiscal adjustment from 2016 onwards.\" Answer: \"tightening\".\nExample 2: Country: China; Year: 2019; Text: \"We concur with staff’s suggestion that there is no need for a further large-scale fiscal stimulus at this moment since the effects of trade tensions have already been factored into this year’s budget.\" Answer: \"no change\".\nExample 3: Country: Singapore; Year: 2019; Text: \"While fiscal policy is focused on medium- to long-term restructuring, the authorities stand ready to provide fiscal stimulus should economic conditions take a significant turn for the worse.\" Answer: \"loosening bias\".'}
explanation_dict = {'staff': 'Note that the near-term policy direction recommended by staff is different in concept from staff\'s projected near-term policy direction of the authorities.',
                    'buff': 'Note that the near-term policy direction planned by the authorities are different in concept from IMF staff\'s recommended or projected near-term policy direction.'}

# Add examples and explanation columns based on type
df_stance['examples'] = df_stance['type'].map(example_dict)
df_stance['explanation'] = df_stance['type'].map(explanation_dict)

In [28]:
# Define column mapping: template placeholder -> dataframe column
# Template uses: {COUNTRY}, {YEAR}, {STAFF_TEXT}, {AUTHORITY_TEXT}
# DataFrame has: 'country', 'year', 'staff', 'buff'
column_mapping = {
    'COUNTRY': 'country',
    'YEAR': 'year',
    'TEXT': 'text',
    'TEXT_AUTHOR': 'type',
    'EXAMPLES': 'examples',
    'EXPLANATION': 'explanation'
}

In [29]:
# Build batch messages from first 5 rows
batch_messages, batch_ids = _build_batch_messages_from_df_flexible(
    df_stance,
    prompt_template,
    column_mapping=column_mapping,
    id_column='index',
    max_text_length=8000
)

In [30]:
print(f"Generated {len(batch_messages)} batch messages")
print(f"Sample IDs: {batch_ids}")
print(f"\nFirst message structure:")
print(f"Number of message parts: {len(batch_messages[0])}")
for i, msg in enumerate(batch_messages[30][:4]):  # Show first 2 parts
    print(f"\nPart {i+1} - Role: {msg.get('role', 'N/A')}")
    content = msg.get('content', '')
    print(f"{content[:3000]}...")
""

Generated 600 batch messages
Sample IDs: ['200', '43', '159', '59', '167', '56', '5', '50', '299', '247', '114', '3', '88', '4', '13', '131', '111', '44', '79', '271', '70', '280', '58', '289', '292', '154', '26', '197', '57', '235', '285', '138', '148', '108', '296', '118', '218', '134', '21', '205', '40', '150', '168', '105', '183', '253', '34', '282', '287', '84', '286', '35', '295', '290', '87', '120', '275', '72', '178', '128', '266', '273', '121', '139', '39', '210', '133', '181', '222', '203', '177', '297', '207', '202', '1', '64', '277', '166', '142', '116', '98', '117', '173', '198', '28', '89', '115', '196', '7', '232', '129', '221', '162', '270', '169', '293', '201', '180', '123', '206', '257', '241', '184', '240', '190', '175', '32', '144', '96', '204', '69', '283', '272', '281', '17', '284', '93', '176', '239', '109', '160', '195', '186', '140', '127', '228', '164', '119', '14', '83', '20', '276', '11', '51', '248', '124', '80', '174', '233', '55', '255', '37', '172', '42'

''

In [31]:
batch_agent = BatchAsyncLLMAgent(
    api_key=os.getenv('OPENAI_API_KEY'),
    model='gpt-5',
    temperature=1
)

In [32]:
# Process a small subset to demo
subset_messages = batch_messages
subset_ids = batch_ids

async def run_batch():
    results = await _process_batch_async(
        batch_agent,
        subset_messages,
        response_model=registry_entry["response_model"],
        batch_size=8,
        max_completion_tokens=2000,
        reasoning_effort='low'
    )
    return results

results = asyncio.run(run_batch())

Processing batches: 100%|██████████| 600/600 [08:09<00:00,  1.23msg/s]


In [33]:
# Show merged outputs (id + parsed response)
merged = _merge_ids_with_responses(subset_ids, results)
print(f"Processed {len(merged)} responses")

# Convert merged results to DataFrame
df_merged = pd.DataFrame(merged)
# Add _llm suffix to column names from the LLM results
llm_columns = [col for col in df_merged.columns if col not in ['id']]
rename_dict = {col: f"{col}_llm" for col in llm_columns}
df_merged = df_merged.rename(columns=rename_dict)
df_merged

Processed 600 responses


,stance_future_llm,id
0,tightening,200
1,tightening,43
2,tightening,159
3,tightening,59
4,loosening,167
...,...,...
595,tightening,9
596,tightening,54
597,no change,19
598,loosening bias,165


In [34]:
# Fix the merge issue by converting id column types to match
df_merged['id'] = df_merged['id'].astype(int)
df_merged = df_merged.merge(df_stance, left_on='id', right_on='index', how='left')
# Keep only records with no NaN values in key columns
df_merged = df_merged.dropna(subset=[ 'stance_future'])
print(len(df_merged))
df_merged.head()

1200


,stance_future_llm,id,index,Print ISBN,country,year,text,stance_future,type,examples,explanation
0,tightening,200,200,9798400231247,Morocco,2022.0,38. The 2023 Budget strikes the right balance ...,tightening,staff,Example 1: Country: Tunisia; Year: 2015; Text:...,Note that the near-term policy direction recom...
1,tightening,200,200,9798400231247,Morocco,2022.0,6. Morocco’s Minister of Finance and BAM signe...,unclear,buff,Example 1: Country: Tunisia; Year: 2015; Text:...,Note that the near-term policy direction plann...
2,tightening,43,43,9781484335918,Papua New Guinea,2017.0,"37. In 2015-16, the Papua New Guinea (PNG) eco...",tightening,staff,Example 1: Country: Tunisia; Year: 2015; Text:...,Note that the near-term policy direction recom...
3,tightening,43,43,9781484335918,Papua New Guinea,2017.0,"In response to these challenges, the newly-ele...",tightening,buff,Example 1: Country: Tunisia; Year: 2015; Text:...,Note that the near-term policy direction plann...
4,tightening,159,159,9781498320993,United States,2019.0,56. The U.S. public debt is on an unsustainabl...,tightening,staff,Example 1: Country: Tunisia; Year: 2015; Text:...,Note that the near-term policy direction recom...


In [35]:
# query v1:
print("Stance Future - Accuracy:", 
      accuracy_score(df_merged['stance_future'], df_merged['stance_future_llm']))
print("Stance Future - F1 Score:", 
      f1_score(df_merged['stance_future'], df_merged['stance_future_llm'], average='weighted'))

print("\nMerging unclear/irrelevant:")
print("Stance Future - Accuracy:", 
      accuracy_score(df_merged['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x), 
                     df_merged['stance_future_llm'].apply(lambda x: 'unclear' if x=='irrelevant' else x)))
print("Stance Future - F1 Score:", 
      f1_score(df_merged['stance_future'].apply(lambda x: 'unclear' if x=='irrelevant' else x), 
               df_merged['stance_future_llm'].apply(lambda x: 'unclear' if x=='irrelevant' else x), average='weighted'))

Stance Future - Accuracy: 0.64
Stance Future - F1 Score: 0.6257594956383206

Merging unclear/irrelevant:
Stance Future - Accuracy: 0.6425
Stance Future - F1 Score: 0.6286072077019849
